In [47]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms 
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

In [48]:
if torch.cuda.is_available():
  device = torch.device("cuda")
  print(f"Using GPU")
elif torch.backends.mps.is_available():
  device = torch.device("mps")
  print(f"Using Apple NPU")
else:
  device = torch.device("cpu")
  print(f"Using CPU")

Using GPU


In [49]:
train_transforms = transforms.Compose(
  [
    transforms.Resize((224, 224)),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
  ]
)

test_transforms = transforms.Compose(
  [
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
  ]
)

In [50]:
full_dataset_for_train = torchvision.datasets.ImageFolder(
  root="labeled_data",
  transform=train_transforms
)

full_dataset_for_test = torchvision.datasets.ImageFolder(
  root="labeled_data",
  transform=test_transforms
)

idxs = np.arange(len(full_dataset_for_train))
labels = full_dataset_for_train.targets

train_idx, test_idx = train_test_split(
  idxs,
  test_size=0.2,
  stratify=labels
)

train_dataset = torch.utils.data.Subset(full_dataset_for_train, train_idx)
test_dataset = torch.utils.data.Subset(full_dataset_for_test, test_idx)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=8, shuffle=False)

In [51]:
_iter = iter(train_loader)

for _ in range(5):
    images, labels = next(_iter)
    display(labels.float().unsqueeze(1))

tensor([[1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [0.],
        [0.],
        [1.]])

tensor([[0.],
        [1.],
        [0.],
        [1.],
        [1.],
        [1.],
        [0.],
        [1.]])

tensor([[1.],
        [1.],
        [0.],
        [1.],
        [1.],
        [0.],
        [0.],
        [1.]])

tensor([[0.],
        [0.],
        [1.],
        [1.],
        [0.],
        [0.],
        [1.],
        [1.]])

tensor([[1.],
        [0.],
        [0.],
        [0.],
        [0.],
        [1.],
        [0.],
        [1.]])

In [52]:
np.sum(np.array(labels) == 0), np.sum(np.array(labels) == 1)

/tmp/ipykernel_87506/2603769112.py:1: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  np.sum(np.array(labels) == 0), np.sum(np.array(labels) == 1)


(np.int64(5), np.int64(3))

In [53]:
np.sum(np.array(labels) == 0) / len(labels), np.sum(np.array(labels) == 1) / len(labels)

/tmp/ipykernel_87506/2479037792.py:1: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  np.sum(np.array(labels) == 0) / len(labels), np.sum(np.array(labels) == 1) / len(labels)


(np.float64(0.625), np.float64(0.375))

In [ ]:
model = nn.Sequential(
  nn.Conv2d(in_channels=3, out_channels=64, kernel_size=7, padding="same"),
  nn.BatchNorm2d(num_features=64),
  nn.ReLU(),
  nn.MaxPool2d(kernel_size=2),
  nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding="same"),
  nn.BatchNorm2d(num_features=128),
  nn.ReLU(),
  nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, padding="same"),
  nn.BatchNorm2d(num_features=128),
  nn.ReLU(),
  nn.MaxPool2d(kernel_size=2),
  nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding="same"),
  nn.BatchNorm2d(num_features=256),
  nn.ReLU(),
  nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, padding="same"),
  nn.BatchNorm2d(num_features=256),
  nn.ReLU(),
  nn.MaxPool2d(kernel_size=2),
  nn.AdaptiveAvgPool2d(output_size=1),
  nn.Flatten(),
  nn.Linear(in_features=256, out_features=128), nn.ReLU(),
  nn.Dropout(p=0.5),
  nn.Linear(in_features=128, out_features=64), nn.ReLU(),
  nn.Dropout(p=0.5),
  nn.Linear(in_features=64, out_features=1)
).to(device)

In [56]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

n_epochs = 50

In [57]:
best_eval_loss = float("inf")

for epoch in range(n_epochs):
  running_loss = 0.0

  model.train()
  for i, data in enumerate(train_loader):
    inputs, labels = data[0].to(device), data[1].to(device).float().unsqueeze(1)

    optimizer.zero_grad()

    outputs = model(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

    running_loss += loss.item()
  
  model.eval()

  with torch.no_grad():
    eval_loss = 0.0
    for i, data in enumerate(test_loader):
      inputs, labels = data[0].to(device), data[1].to(device).float().unsqueeze(1)

      outputs = model(inputs)
      loss = criterion(outputs, labels)
      eval_loss += loss.item()
  
  print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}, Eval Loss: {eval_loss/len(test_loader)}")

  if eval_loss < best_eval_loss:
    best_eval_loss = eval_loss
    torch.save(model.state_dict(), "best_model.pth")

Epoch 1, Loss: 0.6842387047680941, Eval Loss: 0.6898008386294047
Epoch 2, Loss: 0.6661528457294811, Eval Loss: 0.6835635900497437
Epoch 3, Loss: 0.6513360088521783, Eval Loss: 0.6680924495061239
Epoch 4, Loss: 0.6498937444253401, Eval Loss: 0.6395572821299235
Epoch 5, Loss: 0.6356115774674849, Eval Loss: 0.6425326267878214
Epoch 6, Loss: 0.624278957193548, Eval Loss: 0.5794599056243896
Epoch 7, Loss: 0.6241998672485352, Eval Loss: 0.6199110945065817
Epoch 8, Loss: 0.5984083847566084, Eval Loss: 0.5297516882419586
Epoch 9, Loss: 0.6016875017773021, Eval Loss: 0.5567581554253896
Epoch 10, Loss: 0.5764001147313551, Eval Loss: 0.536290854215622
Epoch 11, Loss: 0.592016642743891, Eval Loss: 0.5105442802111307
Epoch 12, Loss: 0.5738077922300859, Eval Loss: 0.5780557294686636
Epoch 13, Loss: 0.541968437758359, Eval Loss: 0.5381985604763031
Epoch 14, Loss: 0.5171678255904805, Eval Loss: 0.48246854543685913
Epoch 15, Loss: 0.5221895927732642, Eval Loss: 0.44131021698315936
Epoch 16, Loss: 0.482

In [58]:
all_predictions = []
all_labels = []

model.eval()

with torch.no_grad():
  for data in test_loader:
    inputs, labels = data[0].to(device), data[1].to(device).float().unsqueeze(1)
    outputs = model(inputs)
    predictions = torch.sigmoid(outputs) > 0.5
    all_predictions.extend(predictions.cpu().numpy())
    all_labels.extend(labels.cpu().numpy())

print(all_predictions)
print(all_labels)
print(np.sum(np.array(all_predictions) == np.array(all_labels).astype(bool)) / len(all_labels))

print(classification_report(all_labels, all_predictions))

[array([False]), array([ True]), array([False]), array([ True]), array([False]), array([False]), array([ True]), array([ True]), array([ True]), array([False]), array([ True]), array([False]), array([False]), array([ True]), array([ True]), array([False]), array([False]), array([ True]), array([ True]), array([ True]), array([False])]
[array([0.], dtype=float32), array([1.], dtype=float32), array([0.], dtype=float32), array([1.], dtype=float32), array([0.], dtype=float32), array([0.], dtype=float32), array([1.], dtype=float32), array([1.], dtype=float32), array([1.], dtype=float32), array([1.], dtype=float32), array([0.], dtype=float32), array([0.], dtype=float32), array([0.], dtype=float32), array([1.], dtype=float32), array([1.], dtype=float32), array([0.], dtype=float32), array([1.], dtype=float32), array([0.], dtype=float32), array([1.], dtype=float32), array([1.], dtype=float32), array([0.], dtype=float32)]
0.8095238095238095
              precision    recall  f1-score   support



In [59]:
torch.save(model.state_dict(), "model.pth")